"""

This script demonstrates training and inference for object detection using Ultralytics YOLOv8 and Ultralytics Yolov9 on a custom Roboflow dataset.

Features:

Visualize Training Images

Train YOLOv8 detection model

Validate YOLOv8 detection model


Train YOLOv9 detection model

Validate YOLOv9 detection model

Run inference on test images with YOLOv9


"""

# Import Libraries

In [ ]:
# import libraries
import zipfile
import requests
import cv2
import matplotlib.pyplot as plt
import glob
import random
import os
import time
from pathlib import Path

In [ ]:
# connect collab workspace to drive if needed
from google.colab import drive
drive.mount('/content/drive')

#INCLUDE PATH TO ROOT OF DOWNLOADED GITHUB FOLDER
REPO_ROOT = Path('')

# Verify and Visualize Dataset

In [ ]:
# check file path and number of images for training, valid, and test data set
!ls -1 "{REPO_ROOT}/Detection/RosetteAugmentedData/train/images" | wc -l
!ls -1 "{REPO_ROOT}/Detection/RosetteAugmentedData/valid/images" | wc -l
!ls -1 "{REPO_ROOT}/Detection/RosetteAugmentedData/test/images" | wc -l

586
32
32


In [ ]:
# converts bounding boxes in YOLO format to xmin, ymin, xmax, ymax
def yolo2bbox(bboxes):
  xmin, ymin = bboxes[0]-bboxes[2]/2, bboxes[1]-bboxes[3]/2
  xmax, ymax = bboxes[0]+bboxes[2]/2, bboxes[1]+bboxes[3]/2
  return xmin, ymin, xmax, ymax

In [ ]:
# function to plot the bounding boxes to their respective images
def plot_box(image, bboxes, labels):
    h, w = image.shape[:2]
    for box, label in zip(bboxes, labels):
        xs = box[::2]
        ys = box[1::2]

        xmin_norm = min(xs)
        ymin_norm = min(ys)
        xmax_norm = max(xs)
        ymax_norm = max(ys)

        xmin = int(xmin_norm * w)
        ymin = int(ymin_norm * h)
        xmax = int(xmax_norm * w)
        ymax = int(ymax_norm * h)

        thickness = max(2, int(w/275))
        cv2.rectangle(
            image,
            (xmin, ymin), (xmax, ymax),
            color=(0, 0, 255),
            thickness=thickness
        )
    return image

In [ ]:
# function to plot images with the bounding boxes
def plot(image_paths, label_paths, num_samples):
    all_images = glob.glob(os.path.join(image_paths, '*.[jJ][pP][gG]'))
    random.shuffle(all_images)

    plt.figure(figsize=(15, 12))

    for i in range(num_samples):
        if i >= len(all_images):
            break

        image_path = all_images[i]
        image = cv2.imread(image_path)
        if image is None:
            continue

        # Get corresponding label
        image_name = os.path.splitext(os.path.basename(image_path))[0]
        label_path = os.path.join(label_paths, f"{image_name}.txt")

        bboxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 11:
                        continue

                    label = parts[0]
                    coords = list(map(float, parts[1:]))
                    bboxes.append(coords)
                    labels.append(label)

        result_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if bboxes:
            result_image = plot_box(result_image, bboxes, labels)

        plt.subplot(2, 2, i+1)
        plt.imshow(result_image)
        plt.axis('off')
        plt.title(image_name)

    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize sample training images
plot(
    image_paths = f"{REPO_ROOT}/Detection/RosetteAugmentedData/train/images",
    label_paths= f"{REPO_ROOT}/Detection/RosetteAugmentedData/train/labels",
    num_samples = 4
)

# Generate YAML file for model training and validation

In [ ]:
# generate a yaml file for the complete dataset
%%writefile rosetteAugmentedData.yaml
path: '{REPO_ROOT}/Detection/RosetteAugmentedData'
train: 'train/images'
val: 'valid/images'

# class names
names:
  0: 'rosette_object'

# Install + Import Ultralytics YOLO for model training and validation

In [ ]:
# install the ultralytics package
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [ ]:
# import YOLO
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# YOLOv8 Detection Model Train, Validate, Test

In [ ]:
# Setting up the hyperparameters
EPOCHS = 25
BATCH = 8
IMG_SIZE = 1280

In [ ]:
# Finetune and train YOLOv8 model.
!yolo \
task = detect \
mode = train \
model = yolov8m.pt \
imgsz = {IMG_SIZE} \
data = /content/rosetteAugmentedData.yaml \
epochs = {EPOCHS} \
batch = {BATCH} \
name = yolov8m_v8_25e_rosetteAugmented

In [ ]:
# Evaluate model
!yolo \
task = detect \
mode = val \
model = {REPO_ROOT}/Detection/Yolov8Detection/yolov8m_v8_25e_rosetteAugmented/weights/best.pt \
name = yolov8m_eval \
data = /content/rosetteAugmentedData.yaml

Ultralytics 8.3.121 🚀 Python-3.11.12 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
100% 755k/755k [00:00<00:00, 22.1MB/s]
val: Fast image access ✅ (ping: 0.8±0.3 ms, read: 0.0±0.0 MB/s, size: 14.0 KB)
val: Scanning /content/drive/MyDrive/Research/PreGithub/RosetteAugmentedDataProperSplit/valid/labels... 32 images, 25 backgrounds, 0 corrupt: 100% 32/32 [00:12<00:00,  2.54it/s]
val: New cache created: /content/drive/MyDrive/Research/PreGithub/RosetteAugmentedDataProperSplit/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% 2/2 [03:49<00:00, 114.74s/it]
                   all         32          8          1      0.846      0.995      0.764
Speed: 55.8ms preprocess, 7074.6ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to runs/detect/yolov8m_eval2
💡 Learn more at https://docs.ultralytics.com/modes/val


In [ ]:
# test model
start = time.time()
!yolo \
task = detect \
mode = predict \
model = {REPO_ROOT}/Detection/Yolov8Detection/yolov8m_v8_25e_rosetteAugmented/weights/best.pt \
source = {REPO_ROOT}/Detection/RosetteAugmentedData/test/images \
imgsz = 1280 \
name = yolov8m_v8_25e_infer1280 \
show_labels = True
end = time.time()
print(f"Total execution time: {end - start:.2f} seconds")

Ultralytics 8.3.121 🚀 Python-3.11.12 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
Model summary (fused): 92 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs

image 1/32 /content/drive/MyDrive/Research/PreGithub/RosetteAugmentedDataProperSplit/test/images/1987-Physco-60k-100nmMOD_jpg.rf.086b0a99c25a0e833815ceb0b022d38e.jpg: 1280x1088 (no detections), 5011.0ms
image 2/32 /content/drive/MyDrive/Research/PreGithub/RosetteAugmentedDataProperSplit/test/images/1987-Physco-60k-100nmMOD_jpg.rf.126eaa1b2c488387087b4952d1061cfc.jpg: 1280x1088 (no detections), 4555.0ms
image 3/32 /content/drive/MyDrive/Research/PreGithub/RosetteAugmentedDataProperSplit/test/images/1987-Physco-60k-100nmMOD_jpg.rf.13ad52a6ccca1398f9fc4ef952c31e15.jpg: 1280x1088 1 rosette_object, 6070.5ms
image 4/32 /content/drive/MyDrive/Research/PreGithub/RosetteAugmentedDataProperSplit/test/images/1987-Physco-60k-100nmMOD_jpg.rf.181d2323db665a1f7cef24c806af96fe.jpg: 1280x1088 (no detections), 4785.7ms
image 5/32 /content/driv

# YOLOv9 Detection Model Train, Validate, Test

In [ ]:
# Setting up the hyperparameters
EPOCHS = 25
BATCH = 8
IMG_SIZE = 640

In [ ]:
# Finetune and train YOLOv9 model.
!yolo \
task = detect \
mode = train \
model = yolov9c.pt \
imgsz = {IMG_SIZE} \
data = /content/rosetteAugmentedData.yaml \
epochs = {EPOCHS} \
batch = {BATCH} \
name = yolov9c_v9_25e_rosetteAugmented

In [ ]:
# evaluate model
!yolo \
task = detect \
mode = val \
model = {REPO_ROOT}/Detection/Yolov9Detection/yolov9c_v9_25e_rosetteAugmented/weights/best.pt \
name = yolov9c_eval \
data = /content/rosetteAugmentedData.yaml

Ultralytics 8.3.121 🚀 Python-3.11.12 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
YOLOv9c summary (fused): 156 layers, 25,320,019 parameters, 0 gradients, 102.3 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 5.3±0.8 MB/s, size: 14.7 KB)
val: Scanning /content/drive/MyDrive/Research/PreGithub/RosetteAugmentedDataProperSplit/valid/labels.cache... 32 images, 25 backgrounds, 0 corrupt: 100% 32/32 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% 2/2 [01:36<00:00, 48.24s/it]
                   all         32          8      0.997          1      0.995      0.854
Speed: 10.5ms preprocess, 2987.9ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to runs/detect/yolov9c_eval
💡 Learn more at https://docs.ultralytics.com/modes/val


In [ ]:
# test model
start = time.time()
!yolo \
task = detect \
mode = predict \
model = {REPO_ROOT}/Detection/Yolov9Detection/yolov9c_v9_25e_rosetteAugmented/weights/best.pt \
source = {REPO_ROOT}/Detection/RosetteAugmentedData/test/images \
imgsz =  {IMG_SIZE} \
name = yolov9c_v9_25e_infer640 \
show_labels = True
end = time.time()
print(f"Total execution time: {end - start:.2f} seconds")

Ultralytics 8.3.121 🚀 Python-3.11.12 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
YOLOv9c summary (fused): 156 layers, 25,320,019 parameters, 0 gradients, 102.3 GFLOPs

image 1/32 /content/drive/MyDrive/Research/PreGithub/RosetteAugmentedDataProperSplit/test/images/1987-Physco-60k-100nmMOD_jpg.rf.086b0a99c25a0e833815ceb0b022d38e.jpg: 640x544 (no detections), 1787.3ms
image 2/32 /content/drive/MyDrive/Research/PreGithub/RosetteAugmentedDataProperSplit/test/images/1987-Physco-60k-100nmMOD_jpg.rf.126eaa1b2c488387087b4952d1061cfc.jpg: 640x544 1 rosette_object, 1768.6ms
image 3/32 /content/drive/MyDrive/Research/PreGithub/RosetteAugmentedDataProperSplit/test/images/1987-Physco-60k-100nmMOD_jpg.rf.13ad52a6ccca1398f9fc4ef952c31e15.jpg: 640x544 1 rosette_object, 2010.5ms
image 4/32 /content/drive/MyDrive/Research/PreGithub/RosetteAugmentedDataProperSplit/test/images/1987-Physco-60k-100nmMOD_jpg.rf.181d2323db665a1f7cef24c806af96fe.jpg: 640x544 1 rosette_object, 2862.1ms
image 5/32 /content/drive/

# Visualize Predictions on Test Images

In [ ]:
# function for plotting and displyaing images and predicted annotations
def visualize(result_dir, num_samples=4) :
  plt.figure(figsize=(20,12))
  image_names = glob.glob(os.path.join(result_dir, '*.jpg'))
  random.shuffle(image_names)
  for i, image_name in enumerate(image_names):
    image = plt.imread(image_name)
    plt.subplot(2, 2, i+1)
    plt.imshow(image)
    plt.axis('off')
    if i == num_samples-1:
      break
  plt.tight_layout()
  plt.show()

In [ ]:
# visualize predicted images
visualize(f"{REPO_ROOT}/Detection/Yolov9Detection/yolov9c_v9_25e_infer640")

<Figure size 2000x1200 with 0 Axes>